# Pipeline de Alfabetização no Brasil — Notebook 5: Qualidade e Monitoramento

**Tech Challenge Fase 2 — FIAP POSTECH**

---

## Objetivo

Executar a suite completa de **qualidade de dados** e **monitoramento da pipeline**:

1. Validações por camada (Bronze, Silver, Gold)
2. Relatório de qualidade com métricas detalhadas
3. Monitoramento operacional (latência, volume, erros)
4. FinOps: estimativa de custo da arquitetura

## Checks Implementados

| Check | Descrição |
|---|---|
| `not_empty` | Tabela não está vazia |
| `no_duplicates` | Sem registros duplicados nas chaves primárias |
| `no_nulls` | Campos obrigatórios sem nulos |
| `range_check` | Valores dentro de limites esperados |
| `ref_integrity` | Chaves estrangeiras com referência válida |
| `completeness` | Cobertura de UFs e municípios esperados |

## 1. Imports e Setup

In [ ]:
import os
import logging
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

try:
    from dotenv import load_dotenv
    load_dotenv(Path("../.env"))
except ImportError:
    pass

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
matplotlib.rcParams["figure.dpi"] = 110

LOCAL_BRONZE = Path("../data/bronze")
LOCAL_SILVER = Path("../data/silver")
LOCAL_GOLD   = Path("../data/gold")
REPORTS_DIR  = Path("../docs/quality_reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
print("Quality Checks iniciado:", RUN_TS)

## 2. Framework de Qualidade

In [ ]:
@dataclass
class CheckResult:
    check: str
    passed: bool
    detail: str = ""
    severity: str = "error"  # error | warning | info


@dataclass
class QualityReport:
    layer: str
    table: str
    run_ts: str = ""
    results: List[CheckResult] = field(default_factory=list)
    n_rows: int = 0

    @property
    def passed(self) -> bool:
        return all(r.passed for r in self.results if r.severity == "error")

    def summary(self) -> str:
        total  = len(self.results)
        ok     = sum(1 for r in self.results if r.passed)
        status = "PASSED" if self.passed else "FAILED"
        return f"[{status}] {self.layer}/{self.table}: {ok}/{total} checks OK | {self.n_rows:,} linhas"

    def to_dict(self) -> dict:
        return {
            "layer"  : self.layer,
            "table"  : self.table,
            "run_ts" : self.run_ts,
            "n_rows" : self.n_rows,
            "passed" : self.passed,
            "results": [{"check": r.check, "passed": r.passed,
                         "detail": r.detail, "severity": r.severity}
                        for r in self.results]
        }


# ── Funções de check ────────────────────────────────────────────────────────

def check_not_empty(df: pd.DataFrame) -> CheckResult:
    passed = len(df) > 0
    return CheckResult("not_empty", passed, f"{len(df):,} linhas")


def check_no_duplicates(df: pd.DataFrame, subset: List[str]) -> CheckResult:
    cols = [c for c in subset if c in df.columns]
    if not cols:
        return CheckResult(f"no_duplicates({subset})", False, "colunas não encontradas")
    dupes = int(df.duplicated(subset=cols).sum())
    return CheckResult(f"no_duplicates({cols})", dupes == 0, f"{dupes:,} duplicatas")


def check_no_nulls(df: pd.DataFrame, cols: List[str]) -> CheckResult:
    present = [c for c in cols if c in df.columns]
    if not present:
        return CheckResult(f"no_nulls({cols})", False, "colunas não encontradas")
    nulls = int(df[present].isnull().sum().sum())
    return CheckResult(f"no_nulls({present})", nulls == 0, f"{nulls:,} nulos")


def check_range(df: pd.DataFrame, col: str, lo: float, hi: float,
                severity: str = "error") -> CheckResult:
    if col not in df.columns:
        return CheckResult(f"range({col})", False, "coluna não encontrada", severity)
    vals = pd.to_numeric(df[col], errors="coerce").dropna()
    violations = int(((vals < lo) | (vals > hi)).sum())
    return CheckResult(
        f"range({col} ∈ [{lo},{hi}])",
        violations == 0,
        f"{violations:,} valores fora do intervalo",
        severity
    )


def check_ref_integrity(df: pd.DataFrame, fk_col: str,
                        ref_values: set) -> CheckResult:
    if fk_col not in df.columns:
        return CheckResult(f"ref_integrity({fk_col})", False, "coluna não encontrada")
    orphans = int((~df[fk_col].isin(ref_values)).sum())
    return CheckResult(
        f"ref_integrity({fk_col})", orphans == 0,
        f"{orphans:,} sem referência"
    )


def check_completeness_ufs(df: pd.DataFrame, col: str,
                            expected_count: int = 27) -> CheckResult:
    if col not in df.columns:
        return CheckResult(f"completeness_uf({col})", False, "coluna não encontrada", "warning")
    actual = df[col].nunique()
    passed = actual >= expected_count
    return CheckResult(
        f"completeness_uf({col})", passed,
        f"{actual}/{expected_count} UFs cobertas",
        "warning"
    )


print("Framework de qualidade carregado.")

## 3. Helpers de Leitura

In [ ]:
def read_all_parquets(base_dir: Path) -> pd.DataFrame:
    """Lê todos os Parquets de um diretório (recursivo)."""
    frames = []
    for path in base_dir.rglob("*.parquet"):
        df = pd.read_parquet(path)
        for part in path.parts:
            if "=" in part:
                col, val = part.split("=", 1)
                df[col] = val
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def read_gold(name: str) -> pd.DataFrame:
    path = LOCAL_GOLD / f"{name}.parquet"
    if not path.exists():
        return pd.DataFrame()
    return pd.read_parquet(path)


print("Helpers carregados.")

## 4. Suites de Checks

### 4.1 Bronze

In [ ]:
def suite_bronze() -> List[QualityReport]:
    reports = []
    bronze_tables = {
        "meta_brasil"         : ["ano", "rede"],
        "meta_uf"             : ["ano", "sigla_uf", "rede"],
        "meta_municipio"      : ["ano", "id_municipio", "rede"],
        "indicador_uf"        : ["ano", "sigla_uf", "serie", "rede"],
        "indicador_municipio" : ["ano", "id_municipio", "serie", "rede"],
    }

    for table, pk_cols in bronze_tables.items():
        path = LOCAL_BRONZE / f"{table}.parquet"
        report = QualityReport("bronze", table, RUN_TS)

        if not path.exists():
            report.results.append(CheckResult("file_exists", False, f"{path} não encontrado"))
            reports.append(report)
            continue

        df = pd.read_parquet(path)
        report.n_rows = len(df)
        report.results.append(check_not_empty(df))

        if not df.empty:
            data_cols = [c for c in df.columns if not c.startswith("_")]
            df_data = df[data_cols]
            if "taxa_alfabetizacao" in df_data.columns:
                df_data = df_data.copy()
                df_data["taxa_alfabetizacao"] = pd.to_numeric(
                    df_data["taxa_alfabetizacao"], errors="coerce")
                report.results.append(
                    check_range(df_data, "taxa_alfabetizacao", 0, 100, severity="warning")
                )

        reports.append(report)

    return reports


bronze_reports = suite_bronze()
print("Resultados Bronze:")
for r in bronze_reports:
    print(f"  {r.summary()}")

### 4.2 Silver

In [ ]:
def suite_silver() -> List[QualityReport]:
    reports = []

    # Silver: alfabetizacao_municipio
    df = read_all_parquets(LOCAL_SILVER / "alfabetizacao_municipio")
    if "ano" in df.columns:
        df["ano"] = pd.to_numeric(df["ano"], errors="coerce")
    if "taxa_alfabetizacao" in df.columns:
        df["taxa_alfabetizacao"] = pd.to_numeric(df["taxa_alfabetizacao"], errors="coerce")

    r = QualityReport("silver", "alfabetizacao_municipio", RUN_TS)
    r.n_rows = len(df)
    r.results.append(check_not_empty(df))
    if not df.empty:
        r.results.append(check_no_duplicates(df, ["id_municipio", "ano", "serie", "rede"]))
        r.results.append(check_no_nulls(df, ["id_municipio", "ano"]))
        r.results.append(check_range(df, "taxa_alfabetizacao", 0, 100))
        r.results.append(check_completeness_ufs(df, "sigla_uf"))
    reports.append(r)

    # Silver: alfabetizacao_uf
    df_uf = read_all_parquets(LOCAL_SILVER / "alfabetizacao_uf")
    if "taxa_alfabetizacao" in df_uf.columns:
        df_uf["taxa_alfabetizacao"] = pd.to_numeric(df_uf["taxa_alfabetizacao"], errors="coerce")

    r2 = QualityReport("silver", "alfabetizacao_uf", RUN_TS)
    r2.n_rows = len(df_uf)
    r2.results.append(check_not_empty(df_uf))
    if not df_uf.empty:
        r2.results.append(check_no_nulls(df_uf, ["sigla_uf", "ano"]))
        r2.results.append(check_range(df_uf, "taxa_alfabetizacao", 0, 100))
        r2.results.append(check_completeness_ufs(df_uf, "sigla_uf"))
    reports.append(r2)

    return reports


silver_reports = suite_silver()
print("Resultados Silver:")
for r in silver_reports:
    print(f"  {r.summary()}")
    for cr in r.results:
        icon = "✓" if cr.passed else ("⚠" if cr.severity == "warning" else "✗")
        print(f"    {icon} {cr.check} — {cr.detail}")

### 4.3 Gold

In [ ]:
def suite_gold() -> List[QualityReport]:
    reports = []

    checks_config = [
        ("indicador_municipio",  ["id_municipio", "ano"],    ["id_municipio", "ano"]),
        ("evolucao_temporal_uf", ["ano", "sigla_uf"],        ["ano", "sigla_uf"]),
        ("ranking_uf",           ["sigla_uf"],               ["sigla_uf"]),
        ("municipios_risco",     ["id_municipio"],           ["id_municipio"]),
        ("comparativo_nacional", ["ano", "meta_ano", "rede"], ["ano"]),
    ]

    for table, no_null_cols, dedup_cols in checks_config:
        df = read_gold(table)
        r = QualityReport("gold", table, RUN_TS)
        r.n_rows = len(df)

        r.results.append(check_not_empty(df))
        if not df.empty:
            r.results.append(check_no_duplicates(df, dedup_cols))
            r.results.append(check_no_nulls(df, no_null_cols))
            for col in ["taxa_alfabetizacao", "media_taxa", "media_taxa_alfabetizacao"]:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
                    r.results.append(check_range(df, col, 0, 100, severity="warning"))
            if "sigla_uf" in df.columns:
                r.results.append(check_completeness_ufs(df, "sigla_uf", expected_count=1))

        reports.append(r)

    return reports


gold_reports = suite_gold()
print("Resultados Gold:")
for r in gold_reports:
    print(f"  {r.summary()}")
    for cr in r.results:
        icon = "✓" if cr.passed else ("⚠" if cr.severity == "warning" else "✗")
        print(f"    {icon} {cr.check} — {cr.detail}")

## 5. Relatório Consolidado

In [ ]:
all_reports = bronze_reports + silver_reports + gold_reports

# Tabela resumo
summary_rows = []
for r in all_reports:
    total  = len(r.results)
    passed = sum(1 for cr in r.results if cr.passed)
    errors = sum(1 for cr in r.results if not cr.passed and cr.severity == "error")
    warns  = sum(1 for cr in r.results if not cr.passed and cr.severity == "warning")
    summary_rows.append({
        "Camada"      : r.layer.upper(),
        "Tabela"      : r.table,
        "Linhas"      : r.n_rows,
        "Checks OK"   : f"{passed}/{total}",
        "Erros"       : errors,
        "Avisos"      : warns,
        "Status"      : "PASSED" if r.passed else "FAILED",
    })

summary_df = pd.DataFrame(summary_rows)

print("=" * 70)
print("RELATÓRIO FINAL DE QUALIDADE")
print("=" * 70)
display(summary_df)

n_passed = sum(1 for r in all_reports if r.passed)
n_total  = len(all_reports)
print(f"\nTotal: {n_passed}/{n_total} tabelas PASSED")

In [ ]:
# Salva relatório JSON
report_data = {
    "run_ts"   : RUN_TS,
    "summary"  : summary_df.to_dict(orient="records"),
    "details"  : [r.to_dict() for r in all_reports],
    "passed"   : n_passed == n_total,
}

report_path = REPORTS_DIR / f"quality_report_{RUN_TS}.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report_data, f, ensure_ascii=False, indent=2, default=str)

print(f"Relatório salvo em: {report_path}")

## 6. Monitoramento da Pipeline

In [ ]:
print("Monitoramento da Pipeline — Métricas Operacionais")
print("-" * 55)

# Volume de dados por camada
def dir_size_mb(path: Path) -> float:
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1_048_576


for layer, path in [("Bronze", LOCAL_BRONZE), ("Silver", LOCAL_SILVER), ("Gold", LOCAL_GOLD)]:
    if path.exists():
        n_files = sum(1 for _ in path.rglob("*.parquet"))
        size_mb = dir_size_mb(path)
        print(f"  {layer:8s}: {n_files:3d} arquivos Parquet | {size_mb:.2f} MB")
    else:
        print(f"  {layer:8s}: diretório não encontrado")

print()
print("Alertas de qualidade:")
failures = [r for r in all_reports if not r.passed]
if failures:
    for r in failures:
        failed_checks = [cr for cr in r.results if not cr.passed and cr.severity == "error"]
        for cr in failed_checks:
            print(f"  [ALERTA] {r.layer}/{r.table} — {cr.check}: {cr.detail}")
else:
    print("  [OK] Nenhuma falha crítica detectada.")

## 7. FinOps — Análise de Custo da Arquitetura

In [ ]:
print("=" * 60)
print("ESTIMATIVA DE CUSTO MENSAL (AWS)")
print("=" * 60)

# Estimativas baseadas em preços AWS us-east-1 (Jul/2025)
custo_items = [
    ("S3 Standard (Bronze, < 5 GB)",        0.30,  "us$0.023/GB"),
    ("S3 Standard-IA (Bronze 90d+)",         0.10,  "45% mais barato que Standard"),
    ("S3 Standard (Silver + Gold, < 2 GB)",  0.10,  "us$0.023/GB"),
    ("AWS Glue (4 DPU-hora/mês)",            1.76,  "us$0.44/DPU-hora"),
    ("AWS Athena (5 GB queries/mês)",        0.25,  "us$5/TB; Parquet 10x barato"),
    ("Airflow (Docker local)",              0.00,  "open-source, sem custo AWS"),
    ("CloudWatch Logs",                     0.50,  "us$0.50/GB ingestão"),
]

print(f"\n{'Componente':<42} {'Custo/mês':>10}  Observação")
print("-" * 80)
total = 0
for item, custo, obs in custo_items:
    total += custo
    print(f"  {item:<40} US${custo:>6.2f}   {obs}")

print("-" * 80)
print(f"  {'TOTAL ESTIMADO':40} US${total:>6.2f}/mês")

print()
print("Economias aplicadas:")
savings = [
    ("Parquet vs CSV",             "~70% menos armazenamento S3"),
    ("Particionamento ano/UF",     "~90% menos dados lidos pelo Athena"),
    ("Glue Serverless vs EMR",     "sem cluster fixo, paga só pelo uso"),
    ("Athena vs Redshift",         "sem custo fixo de cluster (~$182/mês economizado)"),
    ("Airflow local vs MWAA",      "~$400/mês economizado"),
    ("S3 Lifecycle Bronze→IA",     "~45% redução no custo de dados frios"),
    ("Athena workgroup 1GB limit", "previne queries acidentais caras"),
]
for prática, impacto in savings:
    print(f"  ✓ {prática:<35}: {impacto}")

In [ ]:
# Gráfico de custo
componentes = [item[0].split("(")[0].strip() for item in custo_items if item[1] > 0]
custos      = [item[1] for item in custo_items if item[1] > 0]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(componentes, custos, color=["#3498db", "#2ecc71", "#9b59b6",
                                            "#e67e22", "#e74c3c", "#1abc9c"])
ax.set_xlabel("Custo mensal estimado (US$)")
ax.set_title("Breakdown de Custo AWS — Pipeline Alfabetização")
ax.set_xlim(0, max(custos) * 1.3)
for bar, val in zip(bars, custos):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
            f"US${val:.2f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "custo_aws.png", bbox_inches="tight")
plt.show()
print(f"Total estimado: US${sum(custos):.2f}/mês")

---
## Resumo Final

### Pipeline Completa

```
Notebook 01  →  Bronze: ingestão INEP + metadados
Notebook 02  →  Silver: limpeza + integração das 5 bases
Notebook 03  →  Gold: 5 datasets analíticos + gráficos
Notebook 04  →  Streaming: simulação de eventos em tempo quase real
Notebook 05  →  Quality: validação de todas as camadas + FinOps
```

### Custo Total Estimado
**US$ 3–5/mês** — graças a Parquet, Athena serverless e Airflow local.

### Próximos passos sugeridos
1. Integrar fontes externas (IBGE, Censo Escolar) na camada Silver
2. Treinar modelo preditivo com a feature matrix da Gold
3. Configurar alertas CloudWatch para falhas de ingestão
4. Deploy do DAG Airflow em produção (MWAA ou EC2)